# Win Model Evaluation

Logistic regression on six features: age, previous seasons, times in danger,
players remaining (`final_n`), has advantage, and confessional share (rolling 3).

Overall metrics (single split + CV), then trajectory analysis showing
how the model's ability to identify the winner evolves episode-by-episode.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import plotly.express as px
from scipy import stats
from src.models.win import (
    train_eval_pipeline, cross_validate,
    metrics_by_episode_number, winner_rank_detail,
)
from src.export import season_predictions
from src.features.build import build_modeling_table
from src.load import load_data
from src.models import win
from src.models.utils import preprocess

import importlib
import src.evaluate as evaluate_mod
importlib.reload(evaluate_mod)

from src.evaluate import (
    calibration_bins,
    cluster_bootstrap_ci,
    loso_finalist_frac1,
    modeling_feature_cols,
    oob_refit_bootstrap,
    summarize_coefficient_stability,
    summarize_univariate_win_associations,
    summarize_winner_margins,
    summarize_winner_picks,
)
from scipy.stats import spearmanr

In [ ]:
data = load_data()
df = build_modeling_table(data)

## Overall metrics

Single train/test split (seasons 1-40 → 41-49) and expanding-window CV.
These give one number averaging over all episodes equally.

In [ ]:
print("=== Single split ===\n")
results = train_eval_pipeline(df)

print("\n\n=== Cross-validation ===\n")
cv_results = cross_validate(df)

## Accuracy trajectory (cross-validated)

In [ ]:
# Pool predictions from all CV folds (non-overlapping test seasons → ~49 seasons total)
all_cv_preds = pd.concat([fold["predictions"] for fold in cv_results["folds"]])

# Get accuracy metrics at each episode number, averaged across seasons
by_ep_cv = metrics_by_episode_number(all_cv_preds)
by_ep_cv = by_ep_cv[by_ep_cv["n_seasons"] >= 10]  # drop noisy tail episodes

MODEL_COLOR = "#0d9488"
BASELINE_COLOR = "#9ca3af"
marker_sizes = by_ep_cv["n_seasons"] * 4  # bigger dot = more seasons behind it
z95 = 1.96

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle("Win model accuracy over season (cross-validated)", fontsize=13, y=1.02)

# Left plot: where does the model rank the winner? (lower = better)
ax = axes[0]
ax.plot(by_ep_cv["episode"], by_ep_cv["mean_winner_rank"], "-",
        color=MODEL_COLOR, alpha=0.6, zorder=2)
ax.scatter(by_ep_cv["episode"], by_ep_cv["mean_winner_rank"], s=marker_sizes,
           color=MODEL_COLOR, label="Model (size = n seasons)", zorder=3)
ax.fill_between(
    by_ep_cv["episode"],
    by_ep_cv["mean_winner_rank"] - z95 * by_ep_cv["se_winner_rank"],
    by_ep_cv["mean_winner_rank"] + z95 * by_ep_cv["se_winner_rank"],
    color=MODEL_COLOR, alpha=0.12, zorder=1,
)
ax.plot(by_ep_cv["episode"], by_ep_cv["mean_baseline_rank"], "s--",
        color=BASELINE_COLOR, alpha=0.6, label="Random baseline")
ax.set_xlabel("Episode")
ax.set_ylabel("Mean winner rank (lower = better)")
ax.set_title("Winner rank")
ax.legend()
ax.invert_yaxis()

# Right plot: how often is the winner the model's #1 pick?
ax = axes[1]
ax.plot(by_ep_cv["episode"], by_ep_cv["top1_accuracy"], "-",
        color=MODEL_COLOR, alpha=0.6, zorder=2)
ax.scatter(by_ep_cv["episode"], by_ep_cv["top1_accuracy"], s=marker_sizes,
           color=MODEL_COLOR, label="Model (size = n seasons)", zorder=3)
ax.fill_between(
    by_ep_cv["episode"],
    (by_ep_cv["top1_accuracy"] - z95 * by_ep_cv["se_top1"]).clip(0),
    (by_ep_cv["top1_accuracy"] + z95 * by_ep_cv["se_top1"]).clip(0, 1),
    color=MODEL_COLOR, alpha=0.12, zorder=1,
)
ax.plot(by_ep_cv["episode"], by_ep_cv["baseline_top1"], "s--",
        color=BASELINE_COLOR, alpha=0.6, label="Random baseline")
ax.set_xlabel("Episode")
ax.set_ylabel("Top-1 accuracy")
ax.set_title("Top-1 accuracy")
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda y, _: f"{y:.0%}"))
ax.legend()

fig.tight_layout()
plt.show()

print(by_ep_cv.to_string(index=False))

## Significance / uncertainty

To judge whether the model genuinely beats random, we rely on the **cluster
bootstrap** further below (the CI on rank improvement), *not* a simple t-test.
A t-test over every season-episode would treat ~500 correlated observations
(episodes within a season move together) as independent and badly overstate
significance. The cell below just builds the per-episode detail that feeds it.

In [ ]:
# Per-(season, episode) detail: winner rank, top-1/3 hits, rank vs random baseline.
# Feeds the cluster bootstrap further below.
detail = winner_rank_detail(all_cv_preds)
print(f"{len(detail)} season-episodes across {detail['season'].nunique()} seasons")

## Winner prediction by season

For each season in the CV predictions, track how the model ranked the eventual
winner across the full season: how often they were the #1 pick, how often they
were in the top 3, their best rank, etc. This uses the app-style framing
("Ranked #1 for X of Y episodes") rather than aggregate accuracy numbers.

In [ ]:
def player_season_summary(preds: pd.DataFrame) -> pd.DataFrame:
    """For every (season, player), compute ranking stats across all their episodes.

    Returns one row per player-season with:
      - n_episodes: how many episodes they appeared in
      - eps_ranked_1, eps_top_3: how many episodes they were ranked #1 / top 3
      - frac_ranked_1, frac_top_3: same as fractions
      - best_rank: their best (lowest) rank in any episode
      - final_rank: their rank in the last episode they appeared in
      - won_season: whether they won
    """
    rng = np.random.default_rng(42)

    rows = []
    for (season, episode), grp in preds.groupby(["season", "episode"]):
        noisy = grp["prob_win"] + rng.uniform(0, 1e-10, len(grp))
        ranks = noisy.rank(ascending=False).astype(int)
        for idx, row in grp.iterrows():
            rows.append({
                "season": season,
                "episode": episode,
                "castaway_id": row["castaway_id"],
                "castaway": row["castaway"],
                "won_season": row["won_season"],
                "rank": ranks[idx],
                "n_players": len(grp),
            })

    episode_ranks = pd.DataFrame(rows)

    summary = episode_ranks.groupby(["season", "castaway_id", "castaway", "won_season"]).agg(
        n_episodes=("episode", "count"),
        eps_ranked_1=("rank", lambda x: (x == 1).sum()),
        eps_top_3=("rank", lambda x: (x <= 3).sum()),
        best_rank=("rank", "min"),
        final_rank=("rank", "last"),
        final_n_players=("n_players", "last"),
    ).reset_index()

    summary["frac_ranked_1"] = summary["eps_ranked_1"] / summary["n_episodes"]
    summary["frac_top_3"] = summary["eps_top_3"] / summary["n_episodes"]

    return summary


summary = player_season_summary(all_cv_preds)
winners = summary[summary["won_season"] == 1].sort_values("season")

print(f"Seasons with CV predictions: {winners['season'].nunique()}\n")
print("=== How the model tracked the eventual winner ===\n")
for _, w in winners.iterrows():
    print(
        f"  S{int(w['season']):>2d}  {w['castaway']:20s}  "
        f"Best rank: #{int(w['best_rank'])}  |  "
        f"Ranked #1 in {int(w['eps_ranked_1']):>2d}/{int(w['n_episodes'])} eps  |  "
        f"Top 3 in {int(w['eps_top_3']):>2d}/{int(w['n_episodes'])} eps  |  "
        f"Final rank: #{int(w['final_rank'])} of {int(w['final_n_players'])}"
    )

print(f"\n--- Aggregate across {len(winners)} seasons ---")
print(f"  Winner was ever ranked #1:           {(winners['eps_ranked_1'] > 0).mean():.0%} of seasons")
print(f"  Winner ranked #1 in final episode:   {(winners['final_rank'] == 1).mean():.0%} of seasons")
print(f"  Winner in top 3 in final episode:    {(winners['final_rank'] <= 3).mean():.0%} of seasons")
print(f"  Mean fraction of episodes at #1:     {winners['frac_ranked_1'].mean():.1%}")
print(f"  Mean fraction of episodes in top 3:  {winners['frac_top_3'].mean():.1%}")
print(f"  Mean best rank achieved:             {winners['best_rank'].mean():.1f}")

In [ ]:
# Compare winners vs other finalists (players present in the final episode).
# The winner naturally survives every episode, so they have more chances to be
# ranked highly. Restricting to finalists controls for that.

final_eps = all_cv_preds.groupby("season")["episode"].max().reset_index()
final_eps.columns = ["season", "max_episode"]

finalists = (
    all_cv_preds
    .merge(final_eps, on="season")
    .query("episode == max_episode")[["season", "castaway_id"]]
)

finalist_summary = summary.merge(finalists, on=["season", "castaway_id"], how="inner")
finalist_summary["is_winner"] = finalist_summary["won_season"].astype(int)

print(f"Finalists across {finalist_summary['season'].nunique()} seasons: "
      f"{len(finalist_summary)} players ({finalist_summary['is_winner'].sum()} winners)\n")

compare = finalist_summary.groupby("is_winner").agg(
    n=("season", "count"),
    mean_frac_ranked_1=("frac_ranked_1", "mean"),
    mean_frac_top_3=("frac_top_3", "mean"),
    mean_best_rank=("best_rank", "mean"),
    mean_eps_ranked_1=("eps_ranked_1", "mean"),
    mean_eps_top_3=("eps_top_3", "mean"),
    pct_ever_ranked_1=("eps_ranked_1", lambda x: (x > 0).mean()),
    mean_final_rank=("final_rank", "mean"),
).rename(index={0: "Other finalists", 1: "Winner"})

print(compare.T.to_string(float_format=lambda x: f"{x:.2f}"))

print("\n\n=== Which metric best separates winners from other finalists? ===\n")

for metric, label in [
    ("frac_ranked_1", "Frac. of eps ranked #1"),
    ("frac_top_3", "Frac. of eps in top 3"),
    ("best_rank", "Best rank (lower=better)"),
    ("final_rank", "Final episode rank"),
]:
    w = finalist_summary[finalist_summary["is_winner"] == 1][metric]
    nw = finalist_summary[finalist_summary["is_winner"] == 0][metric]
    t, p = stats.mannwhitneyu(w, nw, alternative="two-sided")
    direction = "higher" if w.mean() > nw.mean() else "lower"
    if metric in ("best_rank", "final_rank"):
        direction = "better" if w.mean() < nw.mean() else "worse"
    print(f"  {label:35s}  winner={w.mean():.2f}  other={nw.mean():.2f}  "
          f"({direction})  p={p:.4f}")

In [ ]:
# For each season: which finalist prediction method picks the winner?
# Three methods:
#   1. Highest frac of episodes ranked #1 (uses fractions, not raw counts,
#      so returning players like Chris in S38 aren't penalized)
#   2. Highest frac of episodes in top 3
#   3. Ranked #1 in the final episode

results_rows = []
for season, grp in finalist_summary.groupby("season"):
    winner = grp[grp["is_winner"] == 1].iloc[0]
    best_frac_1 = grp.loc[grp["frac_ranked_1"].idxmax()]
    best_frac_3 = grp.loc[grp["frac_top_3"].idxmax()]
    best_finale = grp.loc[grp["final_rank"].idxmin()]

    results_rows.append({
        "season": int(season),
        "winner": winner["castaway"],
        "best_frac_1": best_frac_1["castaway"],
        "best_frac_1_correct": best_frac_1["castaway"] == winner["castaway"],
        "best_frac_3": best_frac_3["castaway"],
        "best_frac_3_correct": best_frac_3["castaway"] == winner["castaway"],
        "finale_pick": best_finale["castaway"],
        "finale_correct": best_finale["castaway"] == winner["castaway"],
        "winner_final_rank": int(winner["final_rank"]),
        "winner_final_n": int(winner["final_n_players"]),
    })

season_results = pd.DataFrame(results_rows)

print("=== Season-level: which method picks the winner among finalists? ===\n")
print(f"  {'':4s}  {'Winner':20s}  {'Highest frac #1':22s}      {'Highest frac top-3':22s}      {'#1 at finale':22s}")
print(f"  {'':4s}  {'':20s}  {'':22s}      {'':22s}      {'(winner rank)'}")
for _, r in season_results.iterrows():
    f1 = "✓" if r["best_frac_1_correct"] else " "
    f3 = "✓" if r["best_frac_3_correct"] else " "
    ff = "✓" if r["finale_correct"] else " "
    print(
        f"  S{r['season']:>2d}  {r['winner']:20s}  "
        f"{r['best_frac_1']:20s} [{f1}]  "
        f"{r['best_frac_3']:20s} [{f3}]  "
        f"{r['finale_pick']:20s} [{ff}]  "
        f"(#{r['winner_final_rank']} of {r['winner_final_n']})"
    )

n = len(season_results)
n_finalists = finalist_summary.groupby("season").size()
baseline = (1 / n_finalists).mean()

print(f"\n--- Prediction rates across {n} seasons (baseline: {baseline:.0%}) ---")
print(f"  Highest frac of eps at #1:  "
      f"{season_results['best_frac_1_correct'].sum()}/{n} "
      f"({season_results['best_frac_1_correct'].mean():.0%})")
print(f"  Highest frac of eps top 3:  "
      f"{season_results['best_frac_3_correct'].sum()}/{n} "
      f"({season_results['best_frac_3_correct'].mean():.0%})")
print(f"  Ranked #1 at finale:        "
      f"{season_results['finale_correct'].sum()}/{n} "
      f"({season_results['finale_correct'].mean():.0%})")

## Confidence intervals on season-level pick rates (Wilson)

Each method above makes **one winner guess per season**, so the result is a
simple proportion over 40 independent seasons (one Bernoulli trial each).
For that, a Wilson confidence interval is the exact, closed-form tool — no
bootstrap needed. (The over-the-season / pooled metrics, where each season
contributes many correlated episodes, need the cluster bootstrap instead;
that comes next.)

In [ ]:
def wilson_ci(n_correct: int, n_total: int, conf: float = 0.95) -> tuple[float, float]:
    """Wilson score confidence interval for a binomial proportion."""
    ci = stats.binomtest(n_correct, n_total).proportion_ci(
        confidence_level=conf, method="wilson"
    )
    return ci.low, ci.high


n = len(season_results)
n_finalists = finalist_summary.groupby("season").size()
baseline = (1 / n_finalists).mean()

methods = [
    ("Highest frac of eps at #1", "best_frac_1_correct"),
    ("Highest frac of eps in top 3", "best_frac_3_correct"),
    ("Ranked #1 at finale", "finale_correct"),
]

print(f"Season-level winner-pick rates with Wilson 95% CIs (n = {n} seasons)")
print(f"Random baseline (1 of ~{n_finalists.mean():.1f} finalists): {baseline:.0%}\n")
print(f"  {'Method':32s}  {'Correct':>8s}  {'Rate':>6s}  {'95% CI':>16s}")
for label, col in methods:
    n_correct = int(season_results[col].sum())
    rate = n_correct / n
    lo, hi = wilson_ci(n_correct, n)
    print(f"  {label:32s}  {n_correct:>5d}/{n:<2d}  {rate:>5.0%}   [{lo:>5.0%}, {hi:>5.0%}]")

print(f"\nNote: all three CIs comfortably exclude the {baseline:.0%} baseline, "
      f"but they are wide — with only {n} seasons, the point estimates carry real uncertainty.")

## Cluster bootstrap CIs on over-the-season metrics

These metrics (top-1 accuracy, mean winner rank, rank improvement) are
**pooled** across all ~500 season-episodes, which are correlated within a
season and so can't be treated as independent. The **cluster bootstrap by
season** resamples whole seasons (keeping each season's episodes together)
with replacement, recomputes each pooled metric thousands of times, and takes
percentile CIs. This respects the within-season correlation and gives honest
error bars — it's the significance/uncertainty result to rely on.

**Caveat on the pooled numbers:** a single pooled figure like ~22% top-1
accuracy averages over a *rising curve* — the model is much weaker early in
the season (large cast) and stronger late (few players left), as the
per-episode trajectory plot above shows. Read the pooled number as a rough
overall summary, not as the model's accuracy at any given point in the season.

In [ ]:

# `detail` (one row per season-episode) was built above.
def pooled_metrics(d: pd.DataFrame) -> dict:
    return {
        "top1_accuracy": d["top1_correct"].mean(),
        "top3_accuracy": d["top3_correct"].mean(),
        "mean_winner_rank": d["winner_rank"].mean(),
        "mean_rank_improvement": d["rank_improvement"].mean(),
    }

boot = cluster_bootstrap_ci(detail, "season", pooled_metrics, n_boot=5000, seed=42)

# Baselines for reference
baseline_top1 = detail["baseline_top1"].mean()
baseline_rank = detail["baseline_rank"].mean()

n_seasons = detail["season"].nunique()
n_obs = len(detail)
print(f"Cluster bootstrap by season ({n_seasons} seasons, {n_obs} season-episodes, 5000 resamples)\n")
print(f"  {'Metric':24s}  {'Estimate':>9s}  {'95% CI':>18s}  {'Baseline':>9s}")

def _fmt(name, key, pct=False, baseline=None):
    b = boot[key]
    if pct:
        est = f"{b['point']:.1%}"
        rng = f"[{b['lo']:.1%}, {b['hi']:.1%}]"
        base = f"{baseline:.1%}" if baseline is not None else ""
    else:
        est = f"{b['point']:.2f}"
        rng = f"[{b['lo']:.2f}, {b['hi']:.2f}]"
        base = f"{baseline:.2f}" if baseline is not None else ""
    print(f"  {name:24s}  {est:>9s}  {rng:>18s}  {base:>9s}")

_fmt("Top-1 accuracy", "top1_accuracy", pct=True, baseline=baseline_top1)
_fmt("Top-3 accuracy", "top3_accuracy", pct=True)
_fmt("Mean winner rank", "mean_winner_rank", baseline=baseline_rank)
_fmt("Mean rank improvement", "mean_rank_improvement", baseline=0.0)

ri = boot["mean_rank_improvement"]
print(f"\nRank improvement vs random (null = 0):")
print(f"  Cluster bootstrap 95% CI: [{ri['lo']:.2f}, {ri['hi']:.2f}]  "
      f"-> {'excludes' if ri['lo'] > 0 else 'includes'} 0")
print(f"  (A naive t-test over all {n_obs} season-episodes would treat them as "
      f"independent and\n   overstate significance; the effective n is closer to "
      f"{n_seasons} seasons, which this bootstrap reflects.)")

## Season predictions

Train both models on all prior seasons, then predict a single target season.
Change `TARGET_SEASON` below to view a different season.

In [ ]:
TARGET_SEASON = 26

preds, _ = season_predictions(df, TARGET_SEASON)

In [ ]:
def plot_season_predictions(preds, prob_col, title, ylabel, highlight_col=None):
    """Interactive line chart of per-player probabilities over episodes."""
    plot_df = preds.copy()
    plot_df["prob_pct"] = (plot_df[prob_col] * 100).round(1)

    # Mark eliminated players in the hover text
    if highlight_col and highlight_col in plot_df.columns:
        plot_df["status"] = plot_df[highlight_col].map({1: "ELIMINATED", 0: ""})
    else:
        plot_df["status"] = ""

    fig = px.line(
        plot_df.sort_values(["castaway", "episode"]),
        x="episode", y=prob_col, color="castaway",
        markers=True,
        title=title,
        labels={prob_col: ylabel, "episode": "Episode", "castaway": "Player"},
        hover_data={"prob_pct": ":.1f", "status": True, prob_col: False},
    )

    # Add X markers for eliminated players
    if highlight_col and highlight_col in plot_df.columns:
        elim = plot_df[plot_df[highlight_col] == 1]
        if not elim.empty:
            fig.add_scatter(
                x=elim["episode"], y=elim[prob_col],
                mode="markers", marker=dict(symbol="x", size=14, color="black", line_width=2),
                name="Eliminated", showlegend=True,
                hoverinfo="skip",
            )

    fig.update_yaxes(tickformat=".0%")
    fig.update_xaxes(dtick=1)
    fig.update_layout(height=500, legend_title_text="Player")
    fig.show()


plot_season_predictions(
    preds, "prob_eliminated",
    title=f"Season {TARGET_SEASON} — Elimination risk by episode",
    ylabel="P(eliminated)",
    highlight_col="eliminated_this_episode",
)

plot_season_predictions(
    preds, "prob_win",
    title=f"Season {TARGET_SEASON} — Win probability by episode",
    ylabel="P(win)",
    highlight_col="eliminated_this_episode",
)

In [ ]:
def cumulative_rank_trajectories(preds: pd.DataFrame) -> pd.DataFrame:
    """For each player at each episode, compute running ranking stats.

    At episode E, for each remaining player: what fraction of episodes 1..E
    were they ranked #1? Top 3? What's their running average rank?
    """
    rng = np.random.default_rng(42)

    rows = []
    for (season, episode), grp in preds.groupby(["season", "episode"]):
        noisy = grp["prob_win"] + rng.uniform(0, 1e-10, len(grp))
        ranks = noisy.rank(ascending=False).astype(int)
        for idx, row in grp.iterrows():
            rows.append({
                "season": season, "episode": episode,
                "castaway_id": row["castaway_id"], "castaway": row["castaway"],
                "won_season": row["won_season"], "prob_win": row["prob_win"],
                "rank": ranks[idx], "n_players": len(grp),
                "eliminated": row.get("eliminated_this_episode", 0),
            })

    ep_ranks = pd.DataFrame(rows).sort_values(["castaway_id", "episode"])

    ep_ranks["is_rank1"] = (ep_ranks["rank"] == 1).astype(int)
    ep_ranks["is_top3"] = (ep_ranks["rank"] <= 3).astype(int)
    ep_ranks["cum_rank1"] = ep_ranks.groupby("castaway_id")["is_rank1"].cumsum()
    ep_ranks["cum_top3"] = ep_ranks.groupby("castaway_id")["is_top3"].cumsum()
    ep_ranks["ep_number"] = ep_ranks.groupby("castaway_id").cumcount() + 1
    ep_ranks["frac_rank1_so_far"] = ep_ranks["cum_rank1"] / ep_ranks["ep_number"]
    ep_ranks["frac_top3_so_far"] = ep_ranks["cum_top3"] / ep_ranks["ep_number"]
    ep_ranks["cum_avg_rank"] = (
        ep_ranks.groupby("castaway_id")["rank"]
        .expanding().mean().reset_index(level=0, drop=True)
    )

    return ep_ranks


season_ranks = cumulative_rank_trajectories(preds)
winner_name = preds.loc[preds["won_season"] == 1, "castaway"].iloc[0]

# Identify top contenders: anyone who was ever ranked #1, plus the winner
ever_rank1 = set(season_ranks.loc[season_ranks["rank"] == 1, "castaway"].unique())
highlight_players = ever_rank1 | {winner_name}
palette = plt.cm.tab10.colors
highlight_colors = {p: palette[i % len(palette)] for i, p in enumerate(sorted(highlight_players - {winner_name}))}
highlight_colors[winner_name] = "#d4a017"

fig, axes = plt.subplots(1, 2, figsize=(16, 6))
fig.suptitle(f"Season {TARGET_SEASON} — Cumulative ranking stats by episode", fontsize=13, y=1.02)

for ax, col, title, ylabel in [
    (axes[0], "frac_rank1_so_far", "Fraction of episodes ranked #1", "Frac. ranked #1 (cumulative)"),
    (axes[1], "frac_top3_so_far", "Fraction of episodes in top 3", "Frac. in top 3 (cumulative)"),
]:
    for castaway, grp in season_ranks.groupby("castaway"):
        is_highlight = castaway in highlight_players
        is_winner = castaway == winner_name
        ax.plot(
            grp["episode"], grp[col],
            color=highlight_colors.get(castaway, "#d4d4d4"),
            alpha=1.0 if is_highlight else 0.15,
            linewidth=2.5 if is_winner else (1.5 if is_highlight else 0.8),
            zorder=10 if is_winner else (5 if is_highlight else 1),
        )
        # Label highlighted players at their last episode
        if is_highlight:
            last = grp.iloc[-1]
            label = f"{castaway} ★" if is_winner else castaway
            ax.annotate(
                label, xy=(last["episode"], last[col]),
                fontsize=8, fontweight="bold" if is_winner else "normal",
                color=highlight_colors[castaway],
                xytext=(5, 0), textcoords="offset points", va="center",
            )

    ax.set_xlabel("Episode")
    ax.set_ylabel(ylabel)
    ax.set_title(title)
    ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda y, _: f"{y:.0%}"))
    ax.set_xlim(season_ranks["episode"].min() - 0.2, season_ranks["episode"].max() + 2.5)

fig.tight_layout()
plt.show()

# Leaderboard: each player's stats as of their last episode
last_ep_per_player = season_ranks.groupby("castaway_id").last().reset_index()
last_ep_per_player = last_ep_per_player.sort_values("frac_rank1_so_far", ascending=False)

print(f"\nAll-season leaderboard (each player's stats through their last episode):\n")
print(f"  {'Player':20s}  {'Last ep':>7s}  {'Frac #1':>8s}  {'Frac top3':>9s}  {'Avg rank':>8s}")
for _, r in last_ep_per_player.iterrows():
    flag = " <-- WINNER" if r["castaway"] == winner_name else ""
    print(
        f"  {r['castaway']:20s}  {int(r['episode']):>5d}    "
        f"{r['frac_rank1_so_far']:>6.0%}    "
        f"{r['frac_top3_so_far']:>6.0%}    "
        f"{r['cum_avg_rank']:>6.1f}  {flag}"
    )

## Bootstrap refits (OOB) — coefficient stability + pick-rate comparison

Descriptive out-of-bag (OOB) refit bootstrap (`oob_refit_bootstrap`). Each
iteration resamples the 50 seasons with replacement, refits the win model once,
and scores the seasons left out of that draw. Future seasons are allowed in
training, so this is **descriptive, not predictive** (temporal CV stays the
source of truth).

**Win model features** (same as `win.FEATURE_COLS` / the app): age,
`num_previous_seasons`, `times_in_danger`, `final_n`, `has_advantage`,
`confessional_share_rolling_3`.

**Primary output:** standardized coefficient vectors across refits → feature
importance with bootstrap CIs (cells below).

**Secondary output:** per-winner pick rate among held-out seasons.

In [ ]:

# Preprocess once with the win model's features, then run the OOB bootstrap.
# B=1000 -> ~1000 refits total (NOT 50*1000); takes ~45s.
boot_df = preprocess(df, win.FEATURE_COLS)

oob = oob_refit_bootstrap(
    boot_df,
    train_fn=lambda d: win.train_model(d),
    predict_fn=lambda m, s, d: win.predict(m, s, d),
    feature_cols=win.FEATURE_COLS,
    n_boot=1000,
    seed=42,
)

picks = summarize_winner_picks(oob, boot_df)

overall = oob["occurrences"]["pick_is_winner"].mean()
print(f"Overall pick rate: {overall:.1%}  "
      f"({len(oob['occurrences'])} OOB season-refits across "
      f"{oob['occurrences']['season'].nunique()} seasons)")
print(f"OOB refits per season (pick_rate denominator): "
      f"{int(picks['n_oob'].min())}-{int(picks['n_oob'].max())}\n")

with pd.option_context("display.max_rows", None):
    display(
        picks.drop(columns="winner_id").style.format({
            "pick_rate": "{:.1%}",
            "mean_rank": "{:.2f}",
            "mean_finalists": "{:.1f}",
        }).hide(axis="index")
    )

In [ ]:
# Winners sorted expected (top) -> against-the-odds (bottom).
order = picks.sort_values("pick_rate")  # ascending so highest ends up at the top
labels = [f"S{int(s)}  {w}" for s, w in zip(order["season"], order["winner"])]
baseline = 100 / order["mean_finalists"].mean()

fig, ax = plt.subplots(figsize=(9, 13))
ax.barh(labels, order["pick_rate"] * 100, color=plt.cm.RdYlGn(order["pick_rate"]))
ax.axvline(baseline, color="#374151", linestyle="--", linewidth=1,
           label=f"Random-finalist baseline ({baseline:.0f}%)")
ax.set_xlabel("% of OOB refits the winner was the model's pick")
ax.set_title("Expected vs. against-the-odds winners (OOB refit bootstrap, B=1000)")
ax.set_xlim(0, 100)
ax.margins(y=0.005)
ax.legend(loc="lower right")
fig.tight_layout()
plt.show()

### Coefficient stability — what makes a winner?

Same OOB refits as above (`oob["coefficients"]`). Each refit stores the **standardized**
logistic-regression coefficients for the six win features listed above. Aggregated per feature:

- **median** and **95% CI** across refits
- **sign_stability** — share of refits with the same sign as the median
- **rank_stability** — share of refits where the feature's |coef| rank is within ±1 of its median rank
- **ci_excludes_zero** — entire 95% CI lies on one side of zero (required for Tier 1)

**reference** = single fit on all seasons (same features as the deployed model). Wide CIs or low sign stability flag collinear or fragile drivers. Associational, not causal.

Tier lists are printed after the plot. **Tier 1 needs both** `ci_excludes_zero` and sign stability ≥ 90%; a strong median with a CI that barely touches zero (e.g. `num_previous_seasons`, ci_lo ≈ −0.01) stays Tier 2.

In [ ]:
ref_model, _ = win.train_model(boot_df)
reference = pd.Series(ref_model.coef_[0], index=win.FEATURE_COLS)

coef_stab = summarize_coefficient_stability(oob, reference_coefs=reference)

display(
    coef_stab.drop(columns="abs_median").style.format({
        "median": "{:+.3f}",
        "ci_lo": "{:+.3f}",
        "ci_hi": "{:+.3f}",
        "reference": "{:+.3f}",
        "sign_stability": "{:.0%}",
        "rank_stability": "{:.0%}",
    }).hide(axis="index")
)

# Forest-style plot: median ± CI, sorted by |median|
plot_df = coef_stab.sort_values("median")
y = np.arange(len(plot_df))
colors = ["#2563eb" if m > 0 else "#dc2626" for m in plot_df["median"]]
xerr = np.array([
    plot_df["median"] - plot_df["ci_lo"],
    plot_df["ci_hi"] - plot_df["median"],
])

fig, ax = plt.subplots(figsize=(8, 5))
ax.errorbar(
    plot_df["median"], y, xerr=xerr, fmt="o", color="#374151",
    ecolor="#9ca3af", capsize=4, markersize=0, zorder=1,
)
ax.scatter(plot_df["median"], y, c=colors, s=80, zorder=2, label="Bootstrap median")
ax.scatter(plot_df["reference"], y, marker="|", c="#111827", s=120, zorder=3,
           label="Single fit (all seasons)")
ax.axvline(0, color="#374151", linestyle="--", linewidth=1)
ax.set_yticks(y)
ax.set_yticklabels(plot_df["feature"])
ax.set_xlabel("Standardized coefficient (positive → higher P(win))")
ax.set_title("Feature stability across OOB refits (error bars = 95% CI)")
ax.legend(loc="lower right", fontsize=9)
fig.tight_layout()
plt.show()

# Tier guide (definitions in next cell; detail in notes/coefficient_stability.md)
def _coef_tier(r):
    if r["ci_excludes_zero"] and r["sign_stability"] >= 0.9:
        return 1
    if r["sign_stability"] >= 0.85:
        return 2
    return 3

tier_labels = {
    1: "Tier 1 — headline drivers (CI excludes 0, stable sign)",
    2: "Tier 2 — discuss with caveats (sign mostly stable; CI may cross 0)",
    3: "Tier 3 — redundant or unstable (do not interpret alone)",
}
print()
for t in [1, 2, 3]:
    names = coef_stab[coef_stab.apply(_coef_tier, axis=1) == t]["feature"].tolist()
    if names:
        print(tier_labels[t] + ":")
        for name in names:
            print(f"  • {name}")
        print()

## Expected vs. against-the-odds — single-fit margin

Fit the model once per season on the other 49 seasons, score the held-out season,
and measure the winner's **season-long-favorite margin**: the fraction of episodes
the winner was the model's #1 pick minus the best other finalist's.

- **margin > 0** → the winner led the field more than any other finalist; larger = more decisively expected.
- **margin < 0** → another finalist led more often; the winner won against the odds.

Unlike the bootstrap pick-rate, this keeps the gradient at the extremes (no saturation at 0/100).
`loso_finalist_frac1` runs one LOSO fit per season; `summarize_winner_margins` aggregates it.

In [ ]:
# One LOSO pass (~50 fits) feeds both winner and loser plots below.
margin_df = preprocess(df, win.FEATURE_COLS)
finalist_frac1 = loso_finalist_frac1(
    margin_df,
    train_fn=lambda d: win.train_model(d),
    predict_fn=lambda m, s, d: win.predict(m, s, d),
)
margins = summarize_winner_margins(finalist_frac1)

with pd.option_context("display.max_rows", None):
    display(
        margins.style.format({
            "winner_frac1": "{:.0%}",
            "best_other_frac1": "{:.0%}",
            "margin": "{:+.0%}",
            "loso_rank": "{:.0f}",
        }).hide(axis="index")
    )

In [ ]:
# Diverging bars: expected (top, large +margin) -> against-the-odds (bottom).
order = margins.sort_values("margin")
labels = [f"S{int(s)}  {w}" for s, w in zip(order["season"], order["winner"])]
norm = (order["margin"] + 1) / 2  # map [-1, 1] onto RdYlGn

fig, ax = plt.subplots(figsize=(9, 13))
ax.barh(labels, order["margin"] * 100, color=plt.cm.RdYlGn(norm))
ax.axvline(0, color="#374151", linestyle="--", linewidth=1,
           label="winner = model's pick (margin > 0)")
ax.set_xlabel("Winner's frac-#1 margin over next finalist (pp)")
ax.set_title("Expected vs. against-the-odds winners (single LOSO fit)")
ax.margins(y=0.005)
ax.legend(loc="lower right")
fig.tight_layout()
plt.show()

In [ ]:
# Compare single-fit margin vs bootstrap pick-rate.

cmp = margins.merge(picks[["season", "pick_rate"]], on="season")
rho = spearmanr(cmp["margin"], cmp["pick_rate"]).correlation

fig, ax = plt.subplots(figsize=(9, 7))
ax.scatter(cmp["margin"] * 100, cmp["pick_rate"] * 100,
           c=plt.cm.RdYlGn(cmp["pick_rate"]), s=70,
           edgecolor="#333", linewidth=0.5, zorder=3)
ax.axvline(0, color="#888", ls="--", lw=1)
ax.axhline(50, color="#888", ls="--", lw=1)
for _, r in cmp.iterrows():
    if 0.15 < r["pick_rate"] < 0.85 or abs(r["margin"]) > 0.5:
        ax.annotate(f"S{int(r['season'])} {r['winner']}",
                    (r["margin"] * 100, r["pick_rate"] * 100),
                    fontsize=7, xytext=(4, 3), textcoords="offset points")
ax.set_xlabel("Single LOSO fit: winner's frac-#1 margin over next finalist (pp)")
ax.set_ylabel("Bootstrap: % of refits winner was the pick")
ax.set_title(f"Single-fit margin vs bootstrap pick-rate  (Spearman \u03c1={rho:.2f})")
fig.tight_layout()
plt.show()

### The model's biggest missed calls — most dominant finalists who *lost*

Same single-fit machinery, flipped to the **non-winning** finalists. 


In [ ]:
# Uses finalist_frac1 from the cell above (no second LOSO pass).
winner_frac = finalist_frac1[finalist_frac1.is_winner == 1].set_index("season")["frac1"]
winner_name = finalist_frac1[finalist_frac1.is_winner == 1].set_index("season")["castaway"]

losers = finalist_frac1[finalist_frac1.is_winner == 0]
top_loser = losers.loc[losers.groupby("season")["frac1"].idxmax()].copy()
top_loser["winner"] = top_loser["season"].map(winner_name)
top_loser["beat_winner"] = top_loser["frac1"] > top_loser["season"].map(winner_frac)

order = top_loser.sort_values("frac1")
labels = [f"S{int(s)}  {c}" for s, c in zip(order["season"], order["castaway"])]
colors = ["#b91c1c" if b else "#9ca3af" for b in order["beat_winner"]]

fig, ax = plt.subplots(figsize=(9, 13))
ax.barh(labels, order["frac1"] * 100, color=colors)
ax.set_xlabel("Top non-winning finalist's frac-#1 (% of episodes led)")
ax.set_title("The model's biggest missed calls: most dominant finalists who lost")
ax.set_xlim(0, 100)
ax.margins(y=0.005)
ax.legend(handles=[plt.Rectangle((0, 0), 1, 1, color="#b91c1c"),
                   plt.Rectangle((0, 0), 1, 1, color="#9ca3af")],
          labels=["model rated them above the actual winner", "winner still led more"],
          loc="lower right")
fig.tight_layout()
plt.show()

### Gender analysis

Two checks on the same LOSO held-out data (Male/Female finalists only):

1. **Margin ranking** — who lands at the top of the expected-vs-against-the-odds bar chart above? Printed gender counts below.
2. **Finale calibration** — when the model assigns a finale P(win), does that bucket's actual win rate match? Table + chart below.

**Calibration chart:** fixed buckets on the x-axis (0–15%, 15–25%, …). Horizontal tick = mean predicted win rate; dot = observed win rate. Aligned = calibrated; dot above tick = underconfident; dot below = overconfident.

In [ ]:

importlib.reload(evaluate_mod)
calibration_bins = evaluate_mod.calibration_bins

def _gender_label(row):
    if row.get("gender_Male", 0) == 1:
        return "Male"
    if row.get("gender_Non-binary", 0) == 1:
        return "Non-binary"
    return "Female"

gender_map = (
    df.groupby(["season", "castaway_id"]).first().apply(_gender_label, axis=1).rename("gender")
)
cal = finalist_frac1.merge(gender_map.reset_index(), on=["season", "castaway_id"])
cal = cal[cal["gender"].isin(["Male", "Female"])]

# Margin ranking by gender
winners_g = margins.merge(
    cal[cal["is_winner"] == 1][["season", "gender"]], on="season",
)
top10 = winners_g.nlargest(10, "margin")
top_q = margins["margin"].quantile(0.75)
top_q_g = winners_g[winners_g["margin"] >= top_q]
print("Margin ranking (top expected winners):")
print(f"  Top 10: {(top10['gender']=='Male').sum()}/10 male")
print(f"  Top quartile (margin >= {top_q:+.0%}): "
      f"{(top_q_g['gender']=='Male').mean():.0%} male  "
      f"(all winners: {(winners_g['gender']=='Male').mean():.0%} male)")
print()

# Finale calibration by gender
bins = calibration_bins(cal, group_col="gender")
display(
    bins[["group", "bin_label", "n", "mean_predicted", "observed_rate"]].style.format({
        "mean_predicted": "{:.1%}",
        "observed_rate": "{:.1%}",
    }).hide(axis="index")
)

# One chart: bucket on x-axis; tick = mean predicted, dot = observed (same data as table).
bins = bins.sort_values(["bin_lo", "group"])
bin_order = (
    bins.drop_duplicates("bin_lo")
    .sort_values("bin_lo")[["bin_label", "bin_lo"]]
    .reset_index(drop=True)
)
x_map = {row.bin_label: i for i, row in bin_order.iterrows()}
offsets = {"Male": -0.12, "Female": 0.12}
styles = {
    "Male": {"color": "#2563eb", "xytext": (-8, 8), "ha": "right", "va": "bottom"},
    "Female": {"color": "#db2777", "xytext": (8, -8), "ha": "left", "va": "top"},
}

fig, ax = plt.subplots(figsize=(10, 6))
for g in ["Male", "Female"]:
    sub = bins[bins["group"] == g]
    for j, (_, r) in enumerate(sub.iterrows()):
        x = x_map[r["bin_label"]] + offsets[g]
        pred, obs = r["mean_predicted"], r["observed_rate"]
        c = styles[g]["color"]
        ax.hlines(pred, x - 0.06, x + 0.06, colors=c, lw=2.5, zorder=2,
                  label="Mean predicted (tick)" if g == "Male" and j == 0 else None)
        ax.scatter(x, obs, color=c, s=70, zorder=3, label=g if j == 0 else None)
        if abs(pred - obs) > 0.01:
            ax.vlines(x, min(pred, obs), max(pred, obs), colors=c, lw=1, alpha=0.4, zorder=1)
        ax.annotate(
            f"n={int(r['n'])}", (x, obs), xytext=styles[g]["xytext"],
            textcoords="offset points", fontsize=8, color=c,
            ha=styles[g]["ha"], va=styles[g]["va"],
        )

ax.set_xticks(range(len(bin_order)))
ax.set_xticklabels(bin_order["bin_label"])
ax.set_xlabel("Finale P(win) bucket")
ax.set_ylabel("Win rate within bucket")
ax.set_title("Calibration by gender: tick = predicted, dot = observed (aligned = calibrated)")
ax.set_ylim(0, 1.0)
ax.set_xlim(-0.5, len(bin_order) - 0.5)
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda y, _: f"{y:.0%}"))
ax.legend(loc="upper left", fontsize=9)
fig.tight_layout()
plt.show()

**Takeaways**

- **Margin:** strongest expected winners skew male (8/10 in top 10; ~77% in top quartile vs ~56% among all winners).
- **Calibration:** mostly well calibrated in the main bins. Standout gap: female 25–35% (predict ~28%, win ~18%).
- **35–50% bucket:** underconfident for both genders, but n = 3–4 — too thin to lean on.
- Margin (season-long frac-#1) and calibration (finale P(win)) are different metrics.

## Univariate associations with winning

One-feature logistic regressions predicting `won_season` on every player-episode row
(same unit as win-model training). 95% CIs from a season-cluster bootstrap. Tribe-status
dummies excluded by default.

This is **marginal** association (one feature at a time), not the conditional effect
inside the full feature win model.

In [ ]:
importlib.reload(evaluate_mod)
from src.evaluate import summarize_univariate_win_associations

uni = summarize_univariate_win_associations(df, n_boot=2000, seed=42)
uni = uni.assign(in_win_model=uni["feature"].isin(win.FEATURE_COLS))

display(
    uni.drop(columns="abs_coef").style.format({
        "coef": "{:+.3f}",
        "ci_lo": "{:+.3f}",
        "ci_hi": "{:+.3f}",
    }).hide(axis="index")
)

# Forest plot: blue = in deployed win model, gray = not
plot_df = uni.sort_values("coef")
y = np.arange(len(plot_df))
colors = ["#2563eb" if m else "#9ca3af" for m in plot_df["in_win_model"]]
xerr = np.array([
    plot_df["coef"] - plot_df["ci_lo"],
    plot_df["ci_hi"] - plot_df["coef"],
])

fig, ax = plt.subplots(figsize=(8, max(5, 0.28 * len(plot_df))))
ax.errorbar(
    plot_df["coef"], y, xerr=xerr, fmt="none",
    ecolor="#9ca3af", capsize=3, zorder=1,
)
ax.scatter(plot_df["coef"], y, c=colors, s=60, zorder=2)
ax.axvline(0, color="#374151", linestyle="--", linewidth=1)
ax.set_yticks(y)
ax.set_yticklabels(plot_df["feature"])
ax.set_xlabel("Marginal standardized coef → P(win)  (one feature at a time)")
ax.set_title("Univariate win associations (all player-episode rows)")
from matplotlib.lines import Line2D
ax.legend(handles=[
    Line2D([0], [0], marker="o", color="w", markerfacecolor="#2563eb", label="In win model"),
    Line2D([0], [0], marker="o", color="w", markerfacecolor="#9ca3af", label="Not in win model"),
], loc="lower right", fontsize=9)
fig.tight_layout()
plt.show()

strong = uni[uni["ci_excludes_zero"]]["feature"].tolist()
print(f"\nMarginal CI excludes 0 ({len(strong)}): {', '.join(strong)}")

**Takeaways (compare to conditional bootstrap tiers above)**

- **Confessional airtime** — strong positive marginally and conditionally.
- **`times_in_danger`** — marginal coef ≈ 0 (CI crosses 0) on all-episode rows; **negative conditionally** in the win model.
- **`individual_immunity_wins`** — modest positive marginally, not in win model.
